# Exploración y selección de modelo

Este notebook acompaña al pipeline (`python -m src.pipeline`) y al cuadro de mando. Lee los artefactos de `data/publicado/`, comprueba que los indicadores tienen sentido y justifica la elección del modelo.

In [1]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = 'notebook_connected'
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
PUB = Path('..') / 'data' / 'publicado'
prov = pd.read_parquet(PUB / 'panel_provincias.parquet')
mun = pd.read_parquet(PUB / 'panel_municipios.parquet')
calidad = json.loads((PUB / 'calidad.json').read_text(encoding='utf-8'))
modelo = json.loads((PUB / 'modelo.json').read_text(encoding='utf-8'))
prov.shape, mun.shape, calidad['ultimo_trimestre_precios'], calidad['ultimo_anio_renta']

((6463, 18), (24857, 19), '2026Q2', 2024)

## 1. Calidad de los datos

Cada tabla, cuántas filas pasan las reglas y por qué se apartan las demás.

In [2]:
pd.DataFrame(calidad['validacion']).T[['desde', 'hasta', 'filas', 'validas', 'rechazadas', 'cobertura_pct']]

,desde,hasta,filas,validas,rechazadas,cobertura_pct
precios_prov,1995Q1,2026Q2,6552,6463,89,98.600
precios_mun,2005Q1,2026Q2,24978,24857,121,94.500
renta_prov,2015,2024,509,508,1,97.700
renta_mun,2015,2024,81203,73777,7426,90.800
poblacion_prov,2015,2024,520,520,0,100.000
poblacion_mun,2015,2024,81292,74611,6681,91.700
ipc,2002Q1,2026Q3,5148,5148,0,100.000
alquiler,2011,2024,672,672,0,100.000
compraventas,2007Q1,2026Q2,4056,4056,0,100.000
tipo_interes,2009Q1,2026Q2,70,70,0,NaN


In [3]:
norm = calidad['normalizacion']
print('Municipios emparejados:', norm['municipios_emparejados'], 'de', norm['municipios_fuente_ministerio'])
print('Métodos:', norm['metodos_municipios'])
pd.DataFrame(norm['municipios_en_otra_provincia'])

Municipios emparejados: 306 de 306
Métodos: {'nombre': 297, 'nombre_otra_provincia': 8, 'parecido': 1}


,provincia_fuente,municipio_fuente,cod_mun
0,Santa Cruz de,Adeje,38001
1,Córdoba,Almuñecar,18017
2,Cuenca,Azuqueca de Henares,19046
3,Guadalajara,Illescas,45081
4,Alicante,Almazora/Almassora,12009
5,Alicante,Benicarló,12027
6,Cáceres,Ames,15002
7,Ourense,Cangas,36008


## 2. El índice de esfuerzo hoy

Último trimestre disponible. La renta de los años sin publicar es estimada.

In [4]:
ultimo = prov[prov['trimestre'] == prov['trimestre'].max()]
ultimo.sort_values('esfuerzo', ascending=False)[['provincia', 'precio_m2', 'renta_hogar', 'esfuerzo', 'diferencial', 'criterio_esfuerzo', 'criterio_precio', 'tensionado']].head(10)

,provincia,precio_m2,renta_hogar,esfuerzo,diferencial,criterio_esfuerzo,criterio_precio,tensionado
3648,Málaga,"3,083.500","40,274.318",0.312,41.718,True,True,True
881,"Balears, Illes","3,980.800","53,678.866",0.302,39.188,True,True,True
3522,Madrid,"4,089.600","58,346.301",0.286,31.809,False,True,True
2517,Gipuzkoa,"3,514.600","55,118.198",0.260,5.481,False,True,True
1007,Barcelona,"3,204.200","52,041.629",0.251,12.046,False,True,True
6031,Bizkaia,"3,024.700","52,828.301",0.233,4.165,False,True,True
377,Alicante/Alacant,"2,020.700","37,472.060",0.220,29.494,False,True,True
4779,Santa Cruz de Tenerife,"2,296.600","42,674.328",0.219,38.695,False,True,True
1385,Cádiz,"2,043.000","38,161.950",0.218,22.532,False,True,True
125,Araba/Álava,"2,531.300","49,447.389",0.209,3.861,False,True,True


In [5]:
fig = px.scatter(ultimo, x='diferencial', y='esfuerzo', hover_name='provincia', color='tensionado',
                 color_discrete_map={True: '#D55E00', False: '#0072B2'},
                 labels={'diferencial': 'Precio − IPC a cinco años (puntos)', 'esfuerzo': 'Esfuerzo'})
fig.add_hline(y=0.30, line_dash='dot'); fig.add_vline(x=3, line_dash='dot')
fig.update_yaxes(tickformat='.0%'); fig.update_layout(height=420, template='simple_white')

La mayoría de provincias tensionadas lo están por el **criterio de precios** (cuadrante derecho), no por el de esfuerzo: los precios llevan cinco años creciendo bastante por encima del IPC, pero la cuota sobre la renta solo supera el 30 % en los mercados más caros.

## 3. Evolución: cuántas provincias cumplen cada criterio

In [6]:
evol = (prov[prov['trimestre'] >= '2015Q1'].groupby('trimestre')[['criterio_esfuerzo', 'criterio_precio', 'tensionado']]
        .sum().reset_index().melt('trimestre', var_name='criterio', value_name='provincias'))
fig = px.line(evol, x='trimestre', y='provincias', color='criterio',
              color_discrete_sequence=['#0072B2', '#E69F00', '#D55E00'])
fig.update_layout(height=380, template='simple_white')

## 4. Modelo: ¿supera a las referencias?

Validación temporal: entrenamiento con trimestres anteriores a 2021, prueba con 2021–2022 (etiquetas de 2023–2024, las últimas calculadas con renta publicada). El umbral de decisión de cada modelo se eligió en la validación temporal dentro del entrenamiento, maximizando F2.

In [7]:
def tabla(nivel):
    m = pd.DataFrame(modelo[nivel]['metricas']).T.drop(columns='matriz')
    return m[[c for c in ['recall', 'precision', 'f2', 'roc_auc', 'umbral', 'f2_validacion_temporal'] if c in m]]
print('Provincias · elegido:', modelo['provincias']['elegido'])
tabla('provincias')

Provincias · elegido: Gradient boosting


,recall,precision,f2,roc_auc,umbral,f2_validacion_temporal
Referencia: clase mayoritaria,0.000,0.000,0.000,NaN,NaN,NaN
Referencia: persistencia,0.783,0.590,0.735,NaN,NaN,0.787
Regresión logística,0.196,1.000,0.233,0.940,0.850,0.631
Bosque aleatorio,0.935,0.277,0.634,0.923,0.350,0.735
Gradient boosting,0.957,0.280,0.645,0.913,0.200,0.741


In [8]:
print('Municipios · elegido:', modelo['municipios']['elegido'])
tabla('municipios')

Municipios · elegido: Regresión logística


,recall,precision,f2,roc_auc,umbral,f2_validacion_temporal
Referencia: clase mayoritaria,1.000,0.614,0.888,NaN,NaN,NaN
Referencia: persistencia,0.897,0.761,0.866,NaN,NaN,0.695
Regresión logística,0.996,0.637,0.895,0.795,0.300,0.929
Bosque aleatorio,0.994,0.641,0.896,0.809,0.350,0.928
Gradient boosting,1.000,0.614,0.888,0.817,0.050,0.927


La **persistencia** (dentro de dos años estará como hoy) es una referencia exigente porque el estado legal tiene mucha inercia: el criterio de precios mira cinco años hacia atrás. Lo que aporta el modelo es detectar los cambios de estado, que la persistencia por definición nunca anticipa.

In [9]:
pd.DataFrame(modelo['provincias']['importancias'])[['descripcion', 'importancia', 'desviacion']].head(8)

,descripcion,importancia,desviacion
0,Nivel de precio (log €/m²),0.177,0.034
1,Precio − IPC a cinco años (puntos),0.053,0.006
2,Crecimiento del alquiler en el año (%),0.040,0.007
3,Crecimiento de la renta en un año (%),0.028,0.007
4,Esfuerzo actual (cuota / renta),0.020,0.005
5,Compraventas por 1.000 habitantes (último año),0.004,0.006
6,Cambio del diferencial en un año (puntos),0.002,0.001
7,Tensionado hoy (0/1),0.000,0.000


In [10]:
pd.DataFrame(modelo['provincias']['coeficientes'])[['descripcion', 'coeficiente']].head(8)

,descripcion,coeficiente
0,Tipo de interés hipotecario (%),2.183
1,Precio − IPC a cinco años (puntos),1.516
2,Nivel de precio (log €/m²),1.246
3,Crecimiento del precio en un año (%),1.145
4,Crecimiento del alquiler en el año (%),0.930
5,Crecimiento de la renta en un año (%),-0.827
6,Compraventas por 1.000 habitantes (último año),0.621
7,Cambio del esfuerzo en un año (puntos),-0.604


## 5. Previsión a dos años

In [11]:
pred = pd.read_parquet(PUB / 'predicciones_provincias.parquet')
pred['escenario'].value_counts()

escenario
Seguirá tensionado      40
Seguirá sin tensión      8
Entrará en tensión       3
Saldrá de la tensión     1
Name: count, dtype: int64

In [12]:
pred[['provincia', 'probabilidad', 'escenario', 'esfuerzo', 'diferencial']].head(12)

,provincia,probabilidad,escenario,esfuerzo,diferencial
0,Ávila,0.995,Seguirá tensionado,0.133,18.176
1,Huelva,0.984,Seguirá tensionado,0.174,18.724
2,"Coruña, A",0.982,Seguirá tensionado,0.164,16.411
3,Santa Cruz de Tenerife,0.981,Seguirá tensionado,0.219,38.695
4,Zaragoza,0.981,Seguirá tensionado,0.176,28.964
5,Segovia,0.978,Seguirá tensionado,0.147,35.593
6,"Balears, Illes",0.978,Seguirá tensionado,0.302,39.188
7,Málaga,0.978,Seguirá tensionado,0.312,41.718
8,Granada,0.978,Seguirá tensionado,0.183,26.936
9,Asturias,0.976,Seguirá tensionado,0.178,17.799
